# Tutorial 2: Simple Retrieval (Lecture 8, Ch. 16)

The introduction notebook already showed a bare `SELECT * FROM vendor`, a single-condition `WHERE`, and a two-condition `WHERE ... AND ...`. This tutorial covers the rest of Lecture 8's retrieval material against the full schema Tutorial 1 built, `DISTINCT`, `ORDER BY`, `OR`, `NOT`, precedence and parentheses, `BETWEEN`, `LIKE`, `IN`, `IS NULL`, and a concatenated-key predicate.

Run this notebook right after Tutorial 1, since it queries the tables that one builds.

In [ ]:
import pymysql

DB_HOST = "mysql"          # use "localhost" if running outside the jupyter container
DB_PORT = 3306
DB_USER = "root"
DB_PASSWORD = "OSUGoPokes"
DB_NAME = "iem4723_mysql_tutorial"

conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
)
cursor = conn.cursor()

## Removing Duplicates with DISTINCT

*item* has twenty rows, but how many actual categories are in there? `SELECT DISTINCT Item_Description` collapses the repeats down to just the unique values, turns out there are only five, Slacks, Shirts, Neckwear, Footwear, and Outerwear.

In [ ]:
cursor.execute("SELECT DISTINCT Item_Description FROM item;")
for row in cursor.fetchall():
    print(row)

## Sorting with ORDER BY

Sort *item* by price, highest first. Adding a second sort column, like *Item_Description* next, would sort within each price whenever two items tie.

In [ ]:
cursor.execute("""
    SELECT Item_ID, Item_Description, Item_Price
    FROM item
    ORDER BY Item_Price DESC, Item_Description ASC;
""")
for row in cursor.fetchall():
    print(row)

## OR and NOT

Remember `AND` from the introduction notebook, requiring every condition to be met? `OR` only needs one of them, here every Footwear or Outerwear item. `NOT` flips a condition entirely, here every item that does not cost five dollars or less.

In [ ]:
cursor.execute("""
    SELECT Item_ID, Item_Description, Item_Price
    FROM item
    WHERE Item_Description = 'Footwear' OR Item_Description = 'Outerwear';
""")
for row in cursor.fetchall():
    print(row)

In [ ]:
cursor.execute("""
    SELECT Item_ID, Item_Description, Item_Price
    FROM item
    WHERE NOT (Item_Price <= 5);
""")
for row in cursor.fetchall():
    print(row)

## Precedence and Parentheses

Every Slacks item that is either priced above twelve dollars, or is stocked in location L3234. Without the parentheses, MySQL would evaluate `AND` before `OR` anyway by default precedence, but writing them out makes the intent obvious at a glance.

In [ ]:
cursor.execute("""
    SELECT Item_ID, Item_Description, Item_Price, Item_StockLocation
    FROM item
    WHERE Item_Description = 'Slacks'
      AND (Item_Price > 12 OR Item_StockLocation = 'L3234');
""")
for row in cursor.fetchall():
    print(row)

## Special Operators: BETWEEN, LIKE, and IN

Three operators that make writing criteria easier. `BETWEEN` checks a range, inclusive on both ends. `LIKE` matches a character pattern, `%` stands in for any number of characters, so `%GROUP%` matches any vendor name containing the word GROUP anywhere in it. `IN` checks membership in a list, shorter than writing the same column with `OR` over and over.

In [ ]:
cursor.execute("""
    SELECT Item_ID, Item_Description, Item_Price
    FROM item
    WHERE Item_Price BETWEEN 5 AND 10;
""")
for row in cursor.fetchall():
    print(row)

In [ ]:
cursor.execute("""
    SELECT Vendor_Num, Vendor_Name
    FROM vendor
    WHERE Vendor_Name LIKE '%GROUP%';
""")
for row in cursor.fetchall():
    print(row)

In [ ]:
cursor.execute("""
    SELECT Item_ID, Item_Description, Item_StockLocation
    FROM item
    WHERE Item_StockLocation IN ('A9309', 'L3234');
""")
for row in cursor.fetchall():
    print(row)

## IS NULL

`IS NULL` checks for a missing value. As it happens, none of our data has any missing values, every order in *customerorder* already has an actual ship date, so this query should come back empty. An empty result here is itself informative, it confirms every order has shipped.

In [ ]:
cursor.execute("""
    SELECT Order_Num, Customer_Num
    FROM customerorder
    WHERE Order_ActualShipDate IS NULL;
""")
rows = cursor.fetchall()
print(f"{len(rows)} orders with no actual ship date.")
for row in rows:
    print(row)

## A Concatenated-Key Predicate

Remember the *dependents* example from lecture, where a concatenated key needed both columns in the predicate, each in its own simple expression? Same idea here. *order_item* has a concatenated primary key, *Order_Num* and *Item_ID* together, so retrieving one specific row needs both.

In [ ]:
cursor.execute("""
    SELECT Order_Num, Item_ID, Item_Qty
    FROM order_item
    WHERE Order_Num = 16639375 AND Item_ID = 'PL638-JD';
""")
for row in cursor.fetchall():
    print(row)

### Closing Up

That covers Lecture 8's retrieval material. Every query here still only touched one table at a time, Tutorial 3 is where joining several of them together finally pays off.

In [ ]:
cursor.close()
conn.close()